# 06 — Prediction Models

**Purpose:** Train and evaluate multiple classifiers to predict High/Low performance from eye-tracking features, with rigorous cross-validation handling for small datasets.

**Input:** `data/processed/dataset_final.parquet`

**Outputs:**
- `outputs/models/trained_model.pkl`
- `outputs/models/model_metadata.json`
- `outputs/reports/classification_report.csv`
- Figures in `outputs/figures/`

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from sklearn.dummy            import DummyClassifier
from sklearn.linear_model     import LogisticRegression
from sklearn.tree             import DecisionTreeClassifier
from sklearn.ensemble         import RandomForestClassifier
from sklearn.svm              import SVC
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection  import (
    StratifiedKFold, LeaveOneOut, cross_validate,
    GridSearchCV, permutation_test_score
)
from sklearn.metrics          import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, RocCurveDisplay,
    ConfusionMatrixDisplay, balanced_accuracy_score
)
from sklearn.pipeline         import Pipeline

from src.config import (
    DATASET_FINAL, OUTPUTS_MODELS, OUTPUTS_REPORTS, OUTPUTS_FIGURES,
    PERFORMANCE_LABEL_COL, RANDOM_STATE, MIN_N_FOR_KFOLD, CV_FOLDS_LARGE
)

sns.set_theme(style='whitegrid')
for p in [OUTPUTS_MODELS, OUTPUTS_REPORTS, OUTPUTS_FIGURES]:
    p.mkdir(parents=True, exist_ok=True)

## 1. Load data

In [ ]:
dataset = pd.read_parquet(DATASET_FINAL)

# Identify scaled feature columns (exclude unscaled copies, metadata, labels)
exclude_cols = [
    'participant_id', PERFORMANCE_LABEL_COL, 'speed_label',
    'accuracy_rate', 'total_score', 'split', 'mean_response_time_ms'
]
task_correct_cols = [c for c in dataset.columns if c.endswith('_correct') and len(c) < 30]
exclude_cols += task_correct_cols
unscaled_cols  = [c for c in dataset.columns if c.endswith('_unscaled')]
exclude_cols  += unscaled_cols

feature_cols = [c for c in dataset.columns if c not in exclude_cols]

print(f"Total participants: {len(dataset)}")
print(f"Feature columns:    {len(feature_cols)}")

X = dataset[feature_cols].values
y = dataset[PERFORMANCE_LABEL_COL].astype(int).values

print(f"Class distribution: High={y.sum()}, Low={(y==0).sum()}")

N = len(y)

## 2. Cross-validation strategy

In [ ]:
if N < MIN_N_FOR_KFOLD:
    print(f"N={N} < {MIN_N_FOR_KFOLD} — using Leave-One-Out CV")
    cv = LeaveOneOut()
    cv_name = 'LOO-CV'
else:
    k = CV_FOLDS_LARGE
    print(f"N={N} >= {MIN_N_FOR_KFOLD} — using Stratified {k}-Fold CV")
    cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)
    cv_name = f'Stratified {k}-Fold CV'

## 3. Define models

In [ ]:
# Feature selection: SelectKBest inside pipeline to prevent leakage
K_FEATURES = min(10, len(feature_cols))  # start with top-10

def make_pipeline(classifier):
    return Pipeline([
        ('select', SelectKBest(f_classif, k=K_FEATURES)),
        ('clf',    classifier)
    ])

models = {
    'Dummy (baseline)': make_pipeline(
        DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
    ),
    'Logistic Regression': make_pipeline(
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, C=1.0)
    ),
    'Decision Tree': make_pipeline(
        DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)
    ),
    'Random Forest': make_pipeline(
        RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
    ),
    'SVM (RBF)': make_pipeline(
        SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)
    ),
}

## 4. Cross-validate all models

In [ ]:
scoring = ['accuracy', 'balanced_accuracy', 'f1_macro', 'roc_auc']
cv_results = {}

for name, model in models.items():
    print(f"Evaluating: {name}...")
    result = cross_validate(model, X, y, cv=cv, scoring=scoring,
                            return_train_score=False, n_jobs=-1)
    cv_results[name] = result

print("\nDone.")

In [ ]:
summary_rows = []
for name, result in cv_results.items():
    row = {'Model': name}
    for metric in scoring:
        scores = result[f'test_{metric}']
        row[f'{metric}_mean'] = scores.mean().round(3)
        row[f'{metric}_std']  = scores.std().round(3)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index('Model')
summary_df.to_csv(OUTPUTS_REPORTS / 'classification_report.csv')
print(f"CV Strategy: {cv_name}")
summary_df

## 5. Visualize CV results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Balanced accuracy comparison
means = summary_df['balanced_accuracy_mean']
stds  = summary_df['balanced_accuracy_std']
colors = ['#bdc3c7'] + ['#3498db'] * (len(means) - 1)  # gray for dummy

axes[0].barh(means.index, means.values, xerr=stds.values,
             color=colors, edgecolor='white', capsize=4)
axes[0].axvline(0.5, color='red', linestyle='--', label='Chance level')
axes[0].set_xlabel('Balanced Accuracy')
axes[0].set_title(f'Balanced Accuracy ({cv_name})')
axes[0].legend()
axes[0].set_xlim(0, 1)

# ROC-AUC comparison
means_roc = summary_df['roc_auc_mean']
stds_roc  = summary_df['roc_auc_std']
axes[1].barh(means_roc.index, means_roc.values, xerr=stds_roc.values,
             color=colors, edgecolor='white', capsize=4)
axes[1].axvline(0.5, color='red', linestyle='--', label='Chance level')
axes[1].set_xlabel('ROC-AUC')
axes[1].set_title(f'ROC-AUC ({cv_name})')
axes[1].legend()
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / '06_model_comparison.png', dpi=150)
plt.show()

## 6. Identify best model and hyperparameter tuning

In [ ]:
# Best model by balanced accuracy (exclude dummy)
non_dummy = summary_df.drop('Dummy (baseline)')
best_name = non_dummy['balanced_accuracy_mean'].idxmax()
print(f"Best model: {best_name} (balanced_accuracy={non_dummy.loc[best_name, 'balanced_accuracy_mean']:.3f})")

# Hyperparameter grids
param_grids = {
    'Logistic Regression': {
        'select__k': [5, 10, min(15, len(feature_cols))],
        'clf__C': [0.01, 0.1, 1.0, 10.0],
        'clf__penalty': ['l1', 'l2'],
        'clf__solver': ['liblinear']
    },
    'Decision Tree': {
        'select__k': [5, 10, min(15, len(feature_cols))],
        'clf__max_depth': [2, 3, 4, 5],
        'clf__min_samples_split': [2, 4]
    },
    'Random Forest': {
        'select__k': [5, 10, min(15, len(feature_cols))],
        'clf__n_estimators': [50, 100, 200],
        'clf__max_depth': [None, 3, 5]
    },
    'SVM (RBF)': {
        'select__k': [5, 10, min(15, len(feature_cols))],
        'clf__C': [0.1, 1.0, 10.0],
        'clf__gamma': ['scale', 'auto']
    }
}

if best_name in param_grids:
    print(f"\nRunning GridSearchCV for {best_name}...")
    grid_search = GridSearchCV(
        models[best_name], param_grids[best_name],
        cv=cv, scoring='balanced_accuracy',
        n_jobs=-1, refit=True
    )
    grid_search.fit(X, y)
    best_model = grid_search.best_estimator_
    print(f"Best params: {grid_search.best_params_}")
    print(f"Best CV balanced_accuracy: {grid_search.best_score_:.3f}")
else:
    best_model = models[best_name]
    best_model.fit(X, y)

## 7. Permutation test (validate above-chance performance)

In [ ]:
print("Running permutation test (1000 permutations)...")
score, perm_scores, p_value = permutation_test_score(
    best_model, X, y,
    scoring='balanced_accuracy',
    cv=cv, n_permutations=1000,
    random_state=RANDOM_STATE, n_jobs=-1
)

print(f"Model CV score:      {score:.3f}")
print(f"Permutation p-value: {p_value:.4f}")
if p_value < 0.05:
    print("Result is significant — performance is above chance.")
else:
    print("WARNING: Performance is NOT significantly above chance. Interpret results cautiously.")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(perm_scores, bins=25, color='steelblue', alpha=0.7, label='Permuted scores')
ax.axvline(score, color='red', linewidth=2, label=f'Observed score={score:.3f}')
ax.set_xlabel('Balanced Accuracy')
ax.set_title(f'Permutation Test — {best_name} (p={p_value:.4f})')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / '06_permutation_test.png', dpi=150)
plt.show()

## 8. Confusion matrix on hold-out test set (if available)

In [ ]:
if 'split' in dataset.columns and (dataset['split'] == 'test').sum() > 0:
    train_mask = dataset['split'] == 'train'
    test_mask  = dataset['split'] == 'test'

    X_train = dataset.loc[train_mask, feature_cols].values
    y_train = dataset.loc[train_mask, PERFORMANCE_LABEL_COL].astype(int).values
    X_test  = dataset.loc[test_mask,  feature_cols].values
    y_test  = dataset.loc[test_mask,  PERFORMANCE_LABEL_COL].astype(int).values

    best_model.fit(X_train, y_train)
    y_pred = best_model.predict(X_test)

    print("Classification report (hold-out test set):")
    print(classification_report(y_test, y_pred, target_names=['Low','High']))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Confusion matrix
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, display_labels=['Low','High'], ax=axes[0], colorbar=False
    )
    axes[0].set_title(f'Confusion Matrix — {best_name}')

    # ROC curve
    if hasattr(best_model, 'predict_proba'):
        y_proba = best_model.predict_proba(X_test)[:, 1]
        RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1], name=best_name)
        axes[1].plot([0,1],[0,1],'k--')
        axes[1].set_title('ROC Curve')

    plt.tight_layout()
    plt.savefig(OUTPUTS_FIGURES / '06_confusion_roc.png', dpi=150)
    plt.show()
else:
    print("No hold-out test set — cross-validation results are the primary evaluation.")

## 9. Save best model and metadata

In [ ]:
# Refit on all data before saving
best_model.fit(X, y)
joblib.dump(best_model, OUTPUTS_MODELS / 'trained_model.pkl')

metadata = {
    'best_model_name':      best_name,
    'cv_strategy':          cv_name,
    'n_participants':       int(N),
    'n_features_total':     len(feature_cols),
    'k_features_selected':  int(K_FEATURES),
    'feature_col_names':    feature_cols,
    'cv_balanced_accuracy': float(score),
    'permutation_p_value':  float(p_value),
    'all_model_results':    summary_df.to_dict(),
}

with open(OUTPUTS_MODELS / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved: {OUTPUTS_MODELS / 'trained_model.pkl'}")
print(f"Saved: {OUTPUTS_MODELS / 'model_metadata.json'}")